In [1]:
# Cell 1: Imports & Config
import os
import json
import base64
import sqlite3
import asyncio
from datetime import datetime
from pathlib import Path

import nest_asyncio
from playwright.async_api import async_playwright
from openai import OpenAI
from dotenv import load_dotenv

# Fix for running asyncio in Jupyter Notebook
nest_asyncio.apply()

# Load .env file
load_dotenv()

# Config
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

SCREENSHOT_DIR = Path("screenshots")
OUTPUTS_DIR = Path("outputs")
SCREENSHOT_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

print("Config loaded")
print(f"API Key found: {bool(OPENAI_API_KEY)}")
print(f"Screenshot folder: {SCREENSHOT_DIR}")
print(f"Outputs folder: {OUTPUTS_DIR}")


Config loaded
API Key found: True
Screenshot folder: screenshots
Outputs folder: outputs


In [2]:
# Cell 2: Database Setup 
## References:  https://sqlite.org/docs.html

def init_db():
    conn = sqlite3.connect("qa_pipeline.db")
    cursor = conn.cursor()

    # Store each pipeline run 
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS runs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            url TEXT NOT NULL,
            username TEXT,
            status TEXT DEFAULT 'pending',
            created_at TEXT
        )
        """
    )

    # Store each page crawled
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS pages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id INTEGER,
            url TEXT,
            screenshot_path TEXT,
            FOREIGN KEY (run_id) REFERENCES runs (id)
        )
        """
    )

    # Store each finding from the Vision Agent
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS findings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id INTEGER,
            page_url TEXT,
            issue_type TEXT,
            description TEXT,
            severity TEXT,
            recommended_fix TEXT,
            FOREIGN KEY (run_id) REFERENCES runs (id)
        )
        """
    )

    # Stores the final generated report 
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS reports (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id INTEGER,
            content TEXT,
            created_at TEXT,
            FOREIGN KEY (run_id) REFERENCES runs (id)
        )
        """
    )

    conn.commit()
    conn.close()
    print("Database initialized")

init_db()

Database initialized


In [3]:
# Cell 3: Crawl Agent
## References: https://playwright.dev/
async def crawl_page(url, username, password, run_id):
    print(f"Starting crawl for URL: {url} as '{username}'...")

    pages_crawled = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False) # for demonstration, headless=True for production!
        page = await browser.new_page()

        # Step 1: Login
        await page.goto(url)
        await page.fill("#user-name", username)
        await page.fill("#password", password)
        await page.click("#login-button")
        await page.wait_for_load_state("networkidle")

        print(f"Logged in as '{username}'")

        # Step 2: Screenshot the inventory page
        screenshot_path = str(SCREENSHOT_DIR / f"{run_id}_inventory.png")
        await page.screenshot(path=screenshot_path, full_page=True)
        pages_crawled.append({"url": page.url, "screenshot": screenshot_path})
        print(f"Screenshot taken for inventory page: {screenshot_path}")

        # Step 3: Visit first product detail page
        await page.click(".inventory_item_name >> nth=0")
        await page.wait_for_load_state("networkidle")
        screenshot_path = str(SCREENSHOT_DIR / f"{run_id}_product.png")
        await page.screenshot(path=screenshot_path, full_page=True)
        pages_crawled.append({"url": page.url, "screenshot": screenshot_path})
        print(f"Screenshot taken for product page: {screenshot_path}")

        # Step 4: Go back and add item to cart
        await page.go_back()
        await page.wait_for_load_state("networkidle")
        await page.click(".btn_inventory >> nth=0")  # Add first item to cart
        print("Added first item to cart")

        # Step 5: Screenshot the cart page
        await page.click(".shopping_cart_link")
        await page.wait_for_load_state("networkidle")
        screenshot_path = str(SCREENSHOT_DIR / f"{run_id}_cart.png")
        await page.screenshot(path=screenshot_path, full_page=True)
        pages_crawled.append({"url": page.url, "screenshot": screenshot_path})
        print(f"Screenshot taken for cart page: {screenshot_path}")

        # Step 6: Checkout form
        await page.click("#checkout")
        await page.wait_for_load_state("networkidle")
        screenshot_path = str(SCREENSHOT_DIR / f"{run_id}_checkout.png")
        await page.screenshot(path=screenshot_path, full_page=True)
        pages_crawled.append({"url": page.url, "screenshot": screenshot_path})
        print(f"Screenshot taken for checkout page: {screenshot_path}")

        await browser.close()

    # Save to DB
    conn = sqlite3.connect("qa_pipeline.db")
    cursor = conn.cursor()
    for p in pages_crawled:
        cursor.execute(
            "INSERT INTO pages (run_id, url, screenshot_path) VALUES (?, ?, ?)",
            (run_id, p["url"], p["screenshot"]),
        )
    conn.commit()
    conn.close()

    print(f"\n Crawl completed - {len(pages_crawled)} pages captured")
    return pages_crawled

# Test run
# Crearte a run record first 
conn = sqlite3.connect("qa_pipeline.db")
cursor = conn.cursor()
cursor.execute(
    "INSERT INTO runs (url, username, status, created_at) VALUES (?, ?, ?, ?)",
    ("https://www.saucedemo.com", "problem_user", "crawling", datetime.now().isoformat())
)
run_id = cursor.lastrowid
conn.commit()
conn.close()

print(f"Run record created with ID: {run_id}")

# Run the crawl agent
pages = asyncio.run(crawl_page("https://www.saucedemo.com", "problem_user", "secret_sauce", run_id))

Run record created with ID: 1
Starting crawl for URL: https://www.saucedemo.com as 'problem_user'...
Logged in as 'problem_user'
Screenshot taken for inventory page: screenshots/1_inventory.png
Screenshot taken for product page: screenshots/1_product.png
Added first item to cart
Screenshot taken for cart page: screenshots/1_cart.png
Screenshot taken for checkout page: screenshots/1_checkout.png

 Crawl completed - 4 pages captured


In [ ]:
# Cell 4: Vision Agent
def encode_image(image_path):
    #Convert screenshot to base64 for the API
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")
    
def vision_agent(pages, run_id):
    print("\nStarting Vision Agent...")
    all_findings = []

    for page in pages:
        print(f"\nAnalyzing: {page['url']}")

        # Encode the screenshot
        image_data = encode_image(page["screenshot"])

        # Send to GPT
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "system",
                    "content": """You are an expert QA engineer specialising in UI/UX 
                                    and functional testing. Analyse the provided screenshot of a web 
                                    application and identify any defects, issues, or areas of concern.

                                    For each issue found, respond in this exact JSON format:
                                    {
                                        "findings": [
                                            {
                                                "issue_type": "visual|functional|ux",
                                                "description": "Clear description of the issue",
                                                "severity": "critical|major|minor",
                                                "location": "Where on the page the issue is",
                                                "recommended_fix": "Specific actionable fix for this issue"
                                            }
                                        ]
                                    }

                                    Be specific and thorough. Look for:
                                    - Broken or incorrect images
                                    - Layout issues or misaligned elements
                                    - Missing or incorrect text
                                    - Broken buttons or interactive elements
                                    - Poor colour contrast or accessibility issues
                                    - Confusing or misleading UI elements
                                    - Form validation issues

                                    Respond ONLY with the JSON, no extra text."""
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": f"Please analyse this screenshot from the page: {page['url']}"
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{image_data}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=1000
        )

        # Parse the response
        raw = response.choices[0].message.content.strip()

        # Strip markdown fences
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        
        result = json.loads(raw)
        findings = result.get("findings", [])

        print(f"Found {len(findings)} issue(s):")
        for f in findings:
            print(f" [{f['severity'].upper()}] {f['issue_type']} - {f['description'][:80]}...")
            f["page_url"] = page["url"]
            all_findings.append(f)

        # Save findings to DB
        conn = sqlite3.connect("qa_pipeline.db")
        cursor = conn.cursor()
        for f in findings:        
            cursor.execute(
                    """INSERT INTO findings
                    (run_id, page_url, issue_type, description, severity, recommended_fix)
                    VALUES (?, ?, ?, ?, ?, ?)""",
                    (run_id, f["page_url"], f["issue_type"], f["description"], 
                    f["severity"], f.get("recommended_fix", ""))
                )
        conn.commit()
        conn.close()

    print(f"Vision Agent complete - {len(all_findings)} total findings saved")
    return all_findings

# Run on pages crawled
findings = vision_agent(pages, run_id)



Starting Vision Agent...

Analyzing: https://www.saucedemo.com/inventory.html
Found 3 issue(s):
 [MAJOR] visual - Identical images used for all product listings....
 [MINOR] functional - The footer contains non-clickable social media icons....
 [MINOR] ux - Company copyright year is set to 2026, which could confuse users....

Analyzing: https://www.saucedemo.com/inventory-item.html?id=5
Found 4 issue(s):
 [MINOR] visual - The product image is cut off, not displaying the complete item....
 [MINOR] visual - The text 'Sauce Labs Fleece Jacket' has inconsistent spacing between 'Sauce' and...
 [MINOR] ux - The back to products link is easy to miss due to its small size and faint color....
 [MINOR] visual - Social media icons have unclear visual contrast which may affect visibility....

Analyzing: https://www.saucedemo.com/cart.html
Found 2 issue(s):
 [MINOR] ux - The product description uses a function name 'carry.allTheThings()' which may co...
 [MAJOR] visual - The footer text and icons 

In [5]:
# Cell 5: Report Agent
def report_agent(findings, run_id, username):
    print("\nStarting Report Agent...")

    # Group findigns by severity for the prompt
    critical = [f for f in findings if f["severity"] == "critical"]
    major = [f for f in findings if f["severity"] == "major"]
    minor = [f for f in findings if f["severity"] == "minor"]

    # Format findings for the prompt
    findings_text = ""
    for f in findings:
        findings_text += f"""
        - [{f['severity'].upper()}] {f['issue_type'].upper()} on {f['page_url']}
        Description: {f['description']}
        Location: {f.get('location', 'Not specified')}
        """

    # Send to GPT
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": """You are a senior QA engineer writing a formal bug report. 
                Generate a clear, structured incident report from the provided findings.
                
                Use this exact structure:
                
                # QA Automated Test Report
                
                ## Summary
                Brief overview of the test run and key findings.
                
                ## Test Details
                - Target Application: 
                - User Role Tested:
                - Pages Analysed:
                - Total Issues Found:
                - Critical: X | Major: X | Minor: X
                
                ## Critical & Major Findings
                For each critical/major issue, include:
                ### [SEVERITY] Issue Title
                - **Type:** visual/functional/ux
                - **Page:** url
                - **Description:** full description
                - **Recommended Fix:** your suggestion
                
                ## Minor Findings
                Brief list of minor issues.
                
                ## Conclusion
                Overall assessment and recommended next steps.
                
                Be professional, specific and actionable."""
            },
            {
                "role": "user",
                "content": f"""Please generate a QA report for the following test run:

        Target: https://www.saucedemo.com
        User Role: {username}
        Total Findings: {len(findings)} ({len(critical)} critical, {len(major)} major, {len(minor)} minor)

        Findings:
{findings_text}"""
            }
        ],
        max_tokens=2000
    )
    
    report_content = response.choices[0].message.content.strip()
    
    # Save to DB
    conn = sqlite3.connect("qa_pipeline.db")
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO reports (run_id, content, created_at) VALUES (?, ?, ?)",
        (run_id, report_content, datetime.now().isoformat())
    )
    
    # Update run status to completed
    cursor.execute(
        "UPDATE runs SET status = 'completed' WHERE id = ?",
        (run_id,)
    )
    conn.commit()
    conn.close()
    
    # Save to markdown file
    report_path = OUTPUTS_DIR / f"report_{run_id}_{username}.md"
    with open(report_path, "w", encoding="utf-8") as file:
        file.write(report_content)
    
    print(f"Report Agent complete")
    print(f"Report saved to {report_path}")
    print(f"\n--- REPORT PREVIEW ---\n")
    print(report_content[:500] + "...")
    
    return report_content, report_path

# Run it
report, report_path = report_agent(findings, run_id, "problem_user")


Starting Report Agent...
Report Agent complete
Report saved to outputs/report_1_problem_user.md

--- REPORT PREVIEW ---

# QA Automated Test Report

## Summary
The automated test run on https://www.saucedemo.com concluded with a total of 14 issues identified across various site pages. These issues include three major concerns related to visual and usability elements, and eleven minor issues primarily concerning visual discrepancies, functionality glitches, and ux improvements.

## Test Details
- Target Application: https://www.saucedemo.com
- User Role Tested: problem_user
- Pages Analysed: Inventory, Cart, Checko...


In [7]:
# Cell 6: HTML Report Generator
def generate_html_report(findings, report_content, run_id, username, pages):
    print("Generating HTML report...")
    
    # Group findings by severity
    critical = [f for f in findings if f["severity"] == "critical"]
    major = [f for f in findings if f["severity"] == "major"]
    minor = [f for f in findings if f["severity"] == "minor"]
    
    # Pre-encode all screenshots to base64
    encoded_screenshots = {}
    for page in pages:
        try:
            encoded_screenshots[page["url"]] = encode_image(page["screenshot"])
        except Exception as e:
            print(f"Could not encode screenshot for {page['url']}: {e}")

    def build_findings_section(group, color, label):
        if not group:
            return f"<h2>{label}</h2><p><em>None found.</em></p>"
        
        cards = ""
        last_page_url = None
        for f in group:
            page_url = f["page_url"]
            
            screenshot_html = ""
            if page_url != last_page_url and page_url in encoded_screenshots:
                img_data = encoded_screenshots[page_url]
                screenshot_html = f"""
                <p><strong>Page:</strong> <code>{page_url}</code></p>
                <img src="data:image/png;base64,{img_data}" 
                     alt="Screenshot of {page_url}"
                     style="width:100%; border: 1px solid #ccc; margin: 8px 0 16px 0;"/>
                """
                last_page_url = page_url

            cards += f"""
            {screenshot_html}
            <div style="border-left: 3px solid {color}; padding: 10px 14px; margin-bottom: 12px; background: #fafafa;">
                <p><strong>[{f['severity'].upper()}]</strong> {f['issue_type'].upper()} — {f['description']}</p>
                <p><small><strong>Location:</strong> {f.get('location', 'Not specified')}</small></p>
                <p><small><strong>Recommended Fix:</strong> {f.get('recommended_fix', 'Not specified')}</small></p>
            </div>
            """
        
        return f"<h2>{label} ({len(group)})</h2>{cards}"

    critical_html = build_findings_section(critical, "#cc0000", "Critical Findings")
    major_html = build_findings_section(major, "#cc6600", "Major Findings")
    minor_html = build_findings_section(minor, "#997700", "Minor Findings")

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>QA Report - {username}</title>
        <style>
            body {{ font-family: Arial, sans-serif; max-width: 960px; margin: 40px auto; padding: 0 20px; color: #222; }}
            h1 {{ border-bottom: 2px solid #222; padding-bottom: 10px; }}
            h2 {{ margin-top: 40px; border-bottom: 1px solid #ccc; padding-bottom: 6px; }}
            table {{ border-collapse: collapse; margin-bottom: 30px; }}
            td, th {{ padding: 6px 14px; border: 1px solid #ccc; font-size: 0.9em; }}
            code {{ background: #f0f0f0; padding: 2px 5px; font-size: 0.85em; }}
            .footer {{ margin-top: 60px; font-size: 0.8em; color: #999; border-top: 1px solid #eee; padding-top: 10px; }}
        </style>
    </head>
    <body>
        <h1>QA Automated Test Report</h1>
        <p>Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")} | User Role: {username}</p>

        <h2>Test Details</h2>
        <table>
            <tr><th>Target Application</th><td>https://www.saucedemo.com</td></tr>
            <tr><th>User Role Tested</th><td>{username}</td></tr>
            <tr><th>Pages Analysed</th><td>{len(set(f['page_url'] for f in findings))}</td></tr>
            <tr><th>Total Issues</th><td>{len(findings)} ({len(critical)} critical, {len(major)} major, {len(minor)} minor)</td></tr>
            <tr><th>Run ID</th><td>{run_id}</td></tr>
        </table>

        {critical_html}
        {major_html}
        {minor_html}

        <div class="footer">Generated by QA Multi-Agent Pipeline | CM3070 Final Project</div>
    </body>
    </html>
    """

    html_path = OUTPUTS_DIR / f"report_{run_id}_{username}.html"
    with open(html_path, "w", encoding="utf-8") as file:
        file.write(html)

    print(f"Report saved to {html_path}")
    print(f"{len(critical)} critical | {len(major)} major | {len(minor)} minor")

    return html_path

# Run it
html_path = generate_html_report(findings, report, run_id, "problem_user", pages)

Generating HTML report...
Report saved to outputs/report_1_problem_user.html
0 critical | 3 major | 11 minor
